In [ ]:
# @title
!pip install opencv-python
import zipfile
from pathlib import Path
from tqdm import tqdm
import cv2
import shutil
import random
import numpy as np
from pathlib import Path

SEED = 993
random.seed(SEED)
np.random.seed(SEED)

ZIP_PATH = "/content/2026-cv-competition.zip"
UNZIP_DIR = "unzipped"
# распаковываем

Path(UNZIP_DIR).mkdir(exist_ok=True, parents=True)

# Using command-line unzip for potentially problematic zip file
!unzip -o {ZIP_PATH} -d {UNZIP_DIR}

print("Готово")

In [3]:
from pathlib import Path

for p in Path(UNZIP_DIR).rglob("*"):
    if p.is_dir() and p.name in ["images", "labels", "test", "train"]:
        print(p)

unzipped/test
unzipped/train
unzipped/test/test
unzipped/train/train
unzipped/test/test/images
unzipped/train/train/labels
unzipped/train/train/images


In [4]:
from pathlib import Path

IMG_DIR = Path(UNZIP_DIR) / "train" / "train" / "images"

imgs = sorted(IMG_DIR.glob("*"))
print("Всего:", len(imgs))

for p in imgs[:5]:
    print(p.name, p.stat().st_size)

Всего: 1697
IMG20240219113638_jpg.rf.ff06b9b93da9ca10600f9c4ababc261f.jpg 88901
IMG20240219113649_jpg.rf.0aa36d5700345b509e0a4b943ae620cd.jpg 70376
IMG20240219113715_jpg.rf.a71fa6e6f4be2b70af832b77db7ee10d.jpg 71800
IMG20240219113726_jpg.rf.55744d06df6f9a980cafa653da6d9b48.jpg 71574
IMG20240219113745_jpg.rf.06494033a07fb1bb98e68899ace88cad.jpg 56567


In [5]:
def add_gaussian_noise(img):
    sigma = random.choice([20, 30, 40, 55])
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    out = img.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)


def add_salt_pepper_noise(img):
    out = img.copy()
    h, w = img.shape[:2]
    amount = random.choice([0.01, 0.02, 0.03, 0.04])
    n = int(amount * h * w)

    ys = np.random.randint(0, h, n)
    xs = np.random.randint(0, w, n)

    out[ys, xs] = random.choice([(0, 0, 0), (5, 5, 5)])

    return out


def add_jpeg_compression(img):
    quality = random.choice([8, 12, 15, 20, 25, 35])
    ok, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not ok:
        return img
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)


def downscale_upscale(img):
    h, w = img.shape[:2]
    scale = random.choice([0.25, 0.30, 0.35, 0.40, 0.50, 0.6])
    small = cv2.resize(
        img,
        (max(1, int(w * scale)), max(1, int(h * scale))),
        interpolation=cv2.INTER_AREA
    )
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def add_blur(img):
    if random.random() < 0.75:
        k = random.choice([9, 11, 13, 15])
        return cv2.GaussianBlur(img, (k, k), 0)
    else:
        k = random.choice([11, 13, 15])
        kernel = np.zeros((k, k), dtype=np.float32)
        if random.random() < 0.5:
            kernel[k // 2, :] = 1.0
        else:
            kernel[:, k // 2] = 1.0
        kernel /= k
        return cv2.filter2D(img, -1, kernel)


def add_scan_lines(img):
    out = img.copy()
    h, w = img.shape[:2]

    for _ in range(random.choice([4, 8, 12, 18])):
        x = random.randint(0, w - 1)
        color = random.choice([(0, 0, 0), (25, 25, 25), (220, 220, 220)])
        thickness = random.choice([1, 1, 2])
        cv2.line(out, (x, 0), (x, h), color, thickness)

    if random.random() < 0.5:
        for _ in range(random.choice([2, 4, 6])):
            y = random.randint(0, h - 1)
            color = random.choice([(0, 0, 0), (40, 40, 40), (230, 230, 230)])
            cv2.line(out, (0, y), (w, y), color, 1)

    return out


def adjust_color_contrast(img):
    alpha = random.uniform(0.45, 1.25)
    beta = random.randint(-35, 35)
    out = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    if random.random() < 0.25:
        gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
        out = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    if random.random() < 0.4:
        hsv = cv2.cvtColor(out, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 1] *= random.uniform(0.4, 1.4)
        hsv[:, :, 2] *= random.uniform(0.7, 1.3)
        hsv = np.clip(hsv, 0, 255).astype(np.uint8)
        out = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return out



def corrupt_like_test(img):
    out = img.copy()

    if random.random() < 0.95:
        out = downscale_upscale(out)

    if random.random() < 0.95:
        out = add_blur(out)

    if random.random() < 0.90:
        out = add_jpeg_compression(out)

    if random.random() < 0.75:
        out = add_gaussian_noise(out)

    if random.random() < 0.85:
        out = add_salt_pepper_noise(out)

    if random.random() < 0.85:
        out = add_scan_lines(out)

    if random.random() < 0.85:
        out = adjust_color_contrast(out)

    return out


In [6]:
from pathlib import Path
import shutil
import cv2

# Define ROOT and related paths
ROOT = Path("/content") # Base path for unzipped data

# UNZIP_DIR is available from previous cells, but we can make it explicit here for clarity if needed.
# For now, rely on it being globally defined.

SRC_ROOT = ROOT / UNZIP_DIR / "train" / "train"
SRC_IMG_DIR = SRC_ROOT / "images"
SRC_LBL_DIR = SRC_ROOT / "labels"

OUT_ROOT = ROOT / "train_full_clean_testnoise"

# Create output directories, similar to what was in the deleted cell
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

(OUT_ROOT / "images/train").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels/train").mkdir(parents=True, exist_ok=True)

# Check if the source image directory exists before proceeding
if not SRC_IMG_DIR.exists():
    raise FileNotFoundError(
        f"Expected image directory not found: {SRC_IMG_DIR}. "
        "This indicates that the zip file extraction might have failed. "
        "Please check the integrity of your '2026-cv-competition.zip' file and ensure it's fully uploaded."
    )

image_paths = sorted([
    p for p in SRC_IMG_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
])

print("Всего исходных картинок:", len(image_paths))


def copy_label(src_img_path, dst_stem):
    label_path = SRC_LBL_DIR / f"{src_img_path.stem}.txt"
    dst_label = OUT_ROOT / "labels/train" / f"{dst_stem}.txt"

    if label_path.exists():
        shutil.copy2(label_path, dst_label)
    else:
        dst_label.write_text("")


bad = 0

for img_path in image_paths:
    img = cv2.imread(str(img_path))

    if img is None:
        print("bad:", img_path)
        bad += 1
        continue

    # clean copy
    shutil.copy2(img_path, OUT_ROOT / "images/train" / img_path.name)
    copy_label(img_path, img_path.stem)

    # noisy copy
    noisy = corrupt_like_test(img)

    noisy_stem = img_path.stem + "_noise"
    noisy_img_path = OUT_ROOT / "images/train" / f"{noisy_stem}.jpg"

    cv2.imwrite(str(noisy_img_path), noisy)
    copy_label(img_path, noisy_stem)

print("bad:", bad)
print("train images:", len(list((OUT_ROOT / "images/train").glob("*"))))
print("train labels:", len(list((OUT_ROOT / "labels/train").glob("*.txt"))))

Всего исходных картинок: 1697
bad: 0
train images: 3394
train labels: 3394


In [7]:
max_cls = -1

for txt_path in (OUT_ROOT / "labels/train").glob("*.txt"):
    for line in txt_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            max_cls = max(max_cls, int(float(parts[0])))

nc = max_cls + 1

yaml_text = f"""path: {OUT_ROOT}
train: images/train
val: images/train

nc: {nc}
names:
"""

for i in range(nc):
    yaml_text += f"  {i}: class_{i}\n"

yaml_path = ROOT / "data_full_clean_testnoise.yaml"
yaml_path.write_text(yaml_text)

print(yaml_path.read_text())

path: /content/train_full_clean_testnoise
train: images/train
val: images/train

nc: 52
names:
  0: class_0
  1: class_1
  2: class_2
  3: class_3
  4: class_4
  5: class_5
  6: class_6
  7: class_7
  8: class_8
  9: class_9
  10: class_10
  11: class_11
  12: class_12
  13: class_13
  14: class_14
  15: class_15
  16: class_16
  17: class_17
  18: class_18
  19: class_19
  20: class_20
  21: class_21
  22: class_22
  23: class_23
  24: class_24
  25: class_25
  26: class_26
  27: class_27
  28: class_28
  29: class_29
  30: class_30
  31: class_31
  32: class_32
  33: class_33
  34: class_34
  35: class_35
  36: class_36
  37: class_37
  38: class_38
  39: class_39
  40: class_40
  41: class_41
  42: class_42
  43: class_43
  44: class_44
  45: class_45
  46: class_46
  47: class_47
  48: class_48
  49: class_49
  50: class_50
  51: class_51



In [8]:
!pip install ultralytics
from ultralytics import YOLO

model = YOLO("yolo11l.pt")

results = model.train(
    data="/content/data_full_clean_testnoise.yaml",
    epochs=20,
    imgsz=640,
    batch=12,
    device=0,
    seed=993,
    workers=2,
    patience=10,
    optimizer="AdamW",
    lr0=0.0008,
    cos_lr=True,
    project="/content/runs",
    name="yolo11m_full_clean_testnoise"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_full_clean_testnoise.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torch

In [10]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

ROOT = Path("/content")

BEST_WEIGHTS = ROOT / "runs/yolo11m_full_clean_testnoise/weights/best.pt"
TEST_IMG_DIR = ROOT / "unzipped/test/test/images"
SAMPLE_PATH = ROOT / "unzipped/sample_submission.csv"  # если лежит в другом месте, поправим
OUT_PATH = ROOT / "submission.csv"

model = YOLO(str(BEST_WEIGHTS))

results = model.predict(
    source=str(TEST_IMG_DIR),
    imgsz=640,
    conf=0.1,
    iou=0.6,
    device=0,
    save=False,
    verbose=False
)

In [11]:
from pathlib import Path
import pandas as pd

preds = {}

for r in results:
    p = Path(r.path)
    image_id = p.stem

    parts = []

    if r.boxes is not None and len(r.boxes) > 0:
        boxes = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        for cls, conf, box in zip(classes, confs, boxes):
            x1, y1, x2, y2 = box

            parts.extend([
                str(cls),
                f"{conf:.6f}",
                str(int(round(x1))),
                str(int(round(y1))),
                str(int(round(x2))),
                str(int(round(y2))),
            ])

    preds[image_id] = " ".join(parts)

fill_value = "0 0.000001 0 0 1 1"

sub = pd.read_csv(SAMPLE_PATH)
sub["PredictionString"] = sub["image_id"].map(preds)

sub["PredictionString"] = (
    sub["PredictionString"]
    .astype("string")
    .replace(["", "nan", "None", "<NA>"], pd.NA)
    .fillna(fill_value)
)

OUT_PATH = ROOT / "submission.csv"
sub.to_csv(OUT_PATH, index=False)

bad = []
for _, row in sub.iterrows():
    n = len(str(row["PredictionString"]).split())
    if n % 6 != 0:
        bad.append((row["image_id"], n))

print("Saved:", OUT_PATH)
print("Плохих строк:", len(bad))
print(sub.head())

Saved: /content/submission.csv
Плохих строк: 0
                                            image_id  \
0  IMG20240228122809_jpg.rf.bb184f37aa98d96f1db1a...   
1  IMG20240228122949_jpg.rf.f763b523e7bb4e7250796...   
2  IMG20240228122955_jpg.rf.18d6c6ad9c8c69153da34...   
3  IMG20240228123115_jpg.rf.3b9ae83d0bc7242bafe52...   
4  IMG20240228123131_jpg.rf.84cf43cf1c7a8d09dc507...   

                                    PredictionString  
0  44 0.839765 382 434 526 548 28 0.604572 374 19...  
1  0 0.653352 227 427 393 549 30 0.569506 4 1 364...  
2  0 0.819999 476 156 609 304 28 0.498307 143 52 ...  
3  43 0.703232 414 330 502 420 30 0.659712 43 0 4...  
4  0 0.446366 455 0 552 163 4 0.418977 0 1 640 63...  
